In [1]:
!pip install "datasets<3.0.0" soundfile

!pip install transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
!pip install resemblyzer  # lightweight speaker embedding library

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 44.0 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73522 sha256=19dc9037f152d6c34ee4f3d17937de8b6b040d7aab44218d092be75951328603
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=c6dc721c0c8130226602262ae6faadc7d049f54919e8126bc5f4357d3982ab55
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built webrtcvad typing


In [22]:
from datasets import load_dataset
ds = load_dataset("google/fleurs", "ur_pk", split='train', trust_remote_code=True)
genders = set(sample["gender"] for sample in ds)

In [23]:
print(genders)

{0, 1}


In [24]:
filtered_samples = []

for sample in ds:
    if sample["gender"] == 0:   # only gender = 0
        filtered_samples.append(sample)

    if len(filtered_samples) >= 30:
        break

print(f"Collected samples: {len(filtered_samples)}")

Collected samples: 30


In [ ]:
#MALE IS 0

In [26]:
print(ds[0])

{'id': 72, 'num_samples': 251520, 'path': '/root/.cache/huggingface/datasets/downloads/extracted/7a048dc78826d0ee1955a6d538b0def012709b27008111ffa1989d1fde7ee572/1001480768452697900.wav', 'audio': {'path': 'train/1001480768452697900.wav', 'array': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
       2.75373459e-05, 7.27176666e-06, 2.99811363e-05]), 'sampling_rate': 16000}, 'transcription': 'مشہور یونانی وُکلاء سیکس کیچاجیوگلو اور جارج نکولاپولس کو کوریڈیلس کی ایھنز کی جیل میں قید کرکے رکھا گیا ہے کیونکہ وہ مکاری اور بدعنوانی کے مرتکب پائے گئے تھے', 'raw_transcription': 'مشہور یونانی وُکلاء، سیکس کیچاجیوگلو اور جارج نکولاپولس کو کوریڈیلس کی ایھنز کی جیل میں قید کرکے رکھا گیا ہے کیونکہ وہ مکاری اور بدعنوانی کے مرتکب پائے گئے تھے۔', 'gender': 1, 'lang_id': 94, 'language': 'Urdu', 'lang_group_id': 4}


In [27]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = filtered_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(0,10):

    text = urdu_conv[i]['transcription']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.03 seconds.
Sample 0: یہ نظریہ اس دعوی کی تردید کرتا ہے کہ چان...
Sample 1: اس کی باقیات سے بیشتر جزیروں میں بارش ہو...
Sample 2: u.s. کے مطابق اسے ایک پوشیدہ ذریعے سے مع...
Waiting for 1.2 minutes...
Sample 3: وہ دن میں اردگرد کی سطح کی نسبت زیادہ ٹھ...
Sample 4: رضاعی نگہداشت کا مقصد ان تمام ضروریات کی...
Sample 5: نیچے سے یہ بہت دشوار دکھائی دیتا ہے اور ...
Waiting for 1.2 minutes...
Sample 6: مینار کا سب سے اوپر حصہ خاص اور مقدس مقا...
Sample 7: سب سے پہلے بجلی ٹھیک کرنے کے لئے سوئچ کو...
Sample 8: پیٹھ پر سامان لاد کر برف پر پھسلنا اس عم...
Waiting for 1.2 minutes...
Sample 9: انیسویں اور بیسویں صدی کے دوران ایک طویل...


In [28]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = filtered_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(10,20):

    text = urdu_conv[i]['transcription']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 10: اس کیس کو ورجینیا میں فائل کیا گیا کیوں ...
Sample 11: سات پوائنٹس کے ساتھ پچھڑتے ہوئے، 2،243 ک...
Waiting for 1.2 minutes...
Sample 12: آج سویرے ہوائؤں کی رفتار تقریباً 83 کلوم...
Sample 13: اکثر و بیشتر طعن و تشنیع کا شکار قاعدہ ث...
Sample 14: خون کے بہاؤ کو جاری رکھنے کے ساتھ-ساتھ م...
Waiting for 1.2 minutes...
Sample 15: ہوا اور لہروں کے ذریعہ فضا میں اڑانے وال...
Sample 16: اس کا مطلب ہے کہ آپ کچھ دنوں کے لیے دن ک...
Sample 17: کچھ ممالک میں پہلی بار جرم سرزد کرنے وال...
Waiting for 1.2 minutes...
Sample 18: تلوار بازی کا جدید کھیل یونیورسٹی میں پڑ...
Sample 19: لونو میں 160-120 کیوبک میٹر ایندھن تھا ج...


In [29]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = filtered_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(20,30):

    text = urdu_conv[i]['transcription']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 20: انسان ہزاروں برسوں سے بڑی بینائی کے لئے ...
Waiting for 1.2 minutes...
Sample 21: زیادہ تر ٹیلی ویژن لوگوں کو خوش کرنے کیل...
Sample 22: بہت سی عمارتیں دیکھنے کے لحاظ سےکافی خوب...
Sample 23: ڈاؤن ہل سنو اسپورٹس جس میں اسکیئنگ اور ا...
Waiting for 1.2 minutes...
Sample 24: میں خوش ہوں کہ ایسے لوگ موجود ہیں جو میر...
Sample 25: سردست کربی میکسلو قلعہ ایک حقیقی قلعے سے...
Sample 26: اب جاپان کے لئے جاپان ایک جزیروں پر مُشت...
Waiting for 1.2 minutes...
Sample 27: تمہارا پاسپورٹ سفر کی تاریخوں سے 6 ماہ ب...
Sample 28: 1995ء میں اسے پارٹیزن کی تاریخ میں بہتری...
Sample 29: اپنے عروج پر، سرطانی طوفان گونو، جسے مال...
Waiting for 1.2 minutes...


In [30]:
# ============================================
# PHASE 2: Package audio files for MUSHRA
# ============================================
import os
import zipfile

# Create folder structure webMUSHRA expects
os.makedirs("mushra_audio/reference", exist_ok=True)
os.makedirs("mushra_audio/generated_meta_mms", exist_ok=True)


for i in range(30):
    # Copy reference
    ref_src = f"ref_{i}.wav"
    gen_src = f"gen_{i}.wav"

    # Read files
    ref_data, ref_sr = sf.read(ref_src)
    gen_data, gen_sr = sf.read(gen_src)

    # Save into structured folders
    sf.write(f"mushra_audio/reference/sample_{i}.wav", ref_data, ref_sr)
    sf.write(f"mushra_audio/generated_meta_mms/sample_{i}.wav", gen_data, gen_sr)


# Zip everything for download
with zipfile.ZipFile("30_mushra_fleurs_audio.zip", "w") as zf:
    for root, dirs, files in os.walk("mushra_audio"):
        for file in files:
            zf.write(os.path.join(root, file))

print("30_mushra_fleurs_audio.zip ready for download!")

# Download in Colab
from google.colab import files
files.download("30_mushra_fleurs_audio.zip")

30_mushra_fleurs_audio.zip ready for download!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>